## Column Generation example

In [1]:
using Pkg

Pkg.activate(".")
Pkg.resolve()

  Activating project at `~/personal/working/ESPPRC_route_generation`
  No Changes to `~/personal/working/ESPPRC_route_generation/Project.toml`
  No Changes to `~/personal/working/ESPPRC_route_generation/Manifest.toml`


In [2]:
using HiGHS
using JuMP
using Graphs
using GraphPlot
using Plots
using LinearAlgebra
using Random

In [17]:
include("pdp.jl")

rota_da_solucao (generic function with 1 method)

## Problema aleatorios 

In [4]:
function problema_aleatorio(altura, largura, m, r, num_caminhoes, L)
    # gerando as cidades aleatorias 
    cid = Array{Float64}(undef, m, 2)
    for i = 1:m 
        cid[i, 1] = largura*rand()
        cid[i, 2] = altura*rand()
    end

    # calculando a distancia entre todas as cidades 
    C = Array{Float64}(undef, m, m)
    for i = 1:m
        C[i, i] = 0
    end
    for t = 1:m-1
        for s = t+1:m
            a = cid[t, 1]
            b = cid[t, 2]
            c = cid[s, 1]
            d = cid[s, 2]
            C[t, s] = sqrt((a-c)^2 + (b-d)^2)
            C[s,t]=C[t, s]
        end
    end

    # gerando uma quantidade r de tarefas 
    task = Array{Int64}(undef, r, 2)
    for i = 1:r
        a = rand(1:m)
        b = rand(1:m)
        while b == a 
            b = rand(1:m)
        end
        task[i, 1] = a
        task[i, 2] = b
    end

    # gerando uma quantidade r de janelas de tempo para realizacao de cada tarefa 
    W = Array{Float64}(undef, r, 2)
    for i = 1:r
        origem = task[i, 1]
        destino = task[i, 2]
        distancia = C[origem, destino]  # distância entre as cidades da tarefa
        c = distancia + (L - distancia) * rand()
        d = c + (L - c) * rand()     
        W[i, 1] = c
        W[i, 2] = d
    end
    return C, task, W
end

problema_aleatorio (generic function with 1 method)

## Grafo

In [5]:
function grafo(C, task, r)
    # Identificar apenas as cidades usadas nas tarefas
    usados = unique(vcat(task[:, 1], task[:, 2]))
    mapa_cidade = Dict(cidade => i for (i, cidade) in enumerate(usados))

    # Criar grafo apenas com as cidades usadas
    g = SimpleDiGraph(length(usados))

    # Criar matriz truncada inicializada com zeros
    C_truncada = zeros(Float64, length(usados), length(usados))

    # Preencher matriz apenas com distâncias das tarefas
    for i = 1:r
        origem_real = task[i, 1]
        destino_real = task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        C_truncada[origem, destino] = trunc(C[origem_real, destino_real])
    end

    # Criar lista de arestas com pesos
    weights = Dict()
    for i in 1:size(task, 1)
        origem_real, destino_real = task[i, 1], task[i, 2]
        origem = mapa_cidade[origem_real]
        destino = mapa_cidade[destino_real]
        if C_truncada[origem, destino] > 0
            add_edge!(g, origem, destino)
            weights[(origem, destino)] = C_truncada[origem, destino]
        end
    end

    # Obter rótulos das arestas
    graph_edges = collect(Graphs.edges(g))
    edge_labels = [weights[(u.src, u.dst)] for u in graph_edges]
    
    # Plotar o grafo (números originais das cidades no rótulo)
    #gplot(g, nodelabel=usados, edgelabel=edge_labels)
    display(gplot(g, nodelabel=usados, edgelabel=edge_labels))
    return g, weights
end

grafo (generic function with 1 method)

## Dados de entrada

In [19]:
# Representacao ficticia do Parana 
altura = 50
largura = 70

# m vai ser a quantidade de cidades dentro do Parana 
m = 10

# r vai ser a quantidade de tarefas que vou ter 
r = 10

# numero de caminhoes
num_caminhoes = 2

# Limite de tempo
L = 200

# Semente 87213 2 3 
numero_primo = 	87213
teste = 2
semente = numero_primo + teste
Random.seed!(semente)

TaskLocalRNG()

In [20]:
C, task, W = problema_aleatorio(altura, largura, m, r, num_caminhoes, L)

([0.0 31.468866593768798 … 21.7907848765921 29.906505347448427; 31.468866593768798 0.0 … 9.73903843553628 24.973395789398815; … ; 21.7907848765921 9.73903843553628 … 0.0 21.368203002187876; 29.906505347448427 24.973395789398815 … 21.368203002187876 0.0], [5 8; 5 8; … ; 10 7; 5 4], [106.32850830863987 139.28964693588722; 167.82878851710015 188.15981215655077; … ; 63.46516792155512 71.34345570620131; 150.12688444391657 195.5159667907481])

In [21]:
A, solucao, g = A_final(C, task, W, num_caminhoes, r)

195.5159667907481

10×10 Matrix{Float64}:
  0.0      31.4689   52.8953  …  24.259   23.4489  21.7908   29.9065
 31.4689    0.0      23.4206     33.788   22.4291   9.73904  24.9734
 52.8953   23.4206    0.0        56.8987  45.3623  31.7696   32.9979
  5.66122  35.4447   57.5907     21.7652  23.7058  26.0096   35.567
 43.5697   27.1183   22.7392     57.1466  47.1265  28.8896   14.9588
 28.399    12.0222   34.1408  …  22.9519  11.3923  12.5627   33.614
 24.259    33.788    56.8987      0.0     11.5711  28.3366   47.2937
 23.4489   22.4291   45.3623     11.5711   0.0     18.368    39.0485
 21.7908    9.73904  31.7696     28.3366  18.368    0.0      21.3682
 29.9065   24.9734   32.9979     47.2937  39.0485  21.3682    0.0

10×2 Matrix{Int64}:
  5   8
  5   8
  1  10
  7   4
  4   6
  8   4
  9  10
  7   2
 10   7
  5   4

10×2 Matrix{Float64}:
 106.329   139.29
 167.829   188.16
 162.509   190.503
  84.8733  139.467
  71.1268  153.219
 128.856   149.942
  45.6151  195.199
 113.882   121.067
  63.4652   71.3435
 150.127   195.516

12×2 Matrix{Float64}:
   0.0     195.516
 106.329   139.29
 167.829   188.16
 162.509   190.503
  84.8733  139.467
  71.1268  153.219
 128.856   149.942
  45.6151  195.199
 113.882   121.067
  63.4652   71.3435
 150.127   195.516
   0.0     195.516

matriz task
matriz Wu
matriz W

Iteração 1

Solução ótima=[0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Valor ótimo=-2.0
Iterações=2
Base=[13, 12, 3, 4, 5, 6, 7, 8, 9, 10, 11]
Zj - Cj (Custo reduzido final) = [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
	Rotulos a serem adicionados:
	Label(-3.0, [149.84121251292174], 12, [1.0, Inf, Inf, Inf, 3.0, 4.0, Inf, 5.0, Inf, 2.0, Inf, 6.0])
	Label(-2.0, [115.91033686994341], 12, [1.0, Inf, Inf, Inf, 3.0, 4.0, Inf, Inf, Inf, 2.0, Inf, 5.0])
	Label(-1.0, [85.23035694859493], 12, [1.0, Inf, Inf, Inf, 3.0, Inf, Inf, Inf, Inf, 2.0, Inf, 4.0])

Iteração 2

Solução ótima=[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
Valor ótimo=-5.0
Iterações=2
Base=[12, 2, 3, 4, 22, 6, 7, 8, 9, 10, 11]
Zj - Cj (Custo reduzido final) = [-1.0, 0.0, 0.0, 0.0, -3.0, 0.0, 0.0, 0

([1.0 0.0 … 1.0 1.0; 0.0 1.0 … 0.0 0.0; … ; 0.0 0.0 … 1.0 1.0; 0.0 0.0 … 0.0 0.0], [0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 11)

In [ ]:
#solucao[g+1:end] (codigo mais atual)